### Preprocessing Libraries

In [ ]:
import numpy as np
import pandas as pd
import re
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

from scipy import spatial
import networkx as nx

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

## Text Preprocessing

### Text Cleaner

In [ ]:
class textCleaner:

    # run all functions in textCleaner
    def clean(self, article):
        sent_tokenized_article = self.textTokenzier(article)
        lemmatized_article = self.textLemmatizer(sent_tokenized_article)
        removed_stopwords_article = self.stopWordsRemover(lemmatized_article)
        return removed_stopwords_article

    def textTokenzier(self, article):
        sentences = sent_tokenize(article)
        sent_tokenized = [simple_preprocess(sentence) for sentence in sentences]
        return sent_tokenized

    def stopWordsRemover(self, sentences):
        stop_words = stopwords.words('english')
        stop_words += ["really", "also", "com", "use", "http", "www", "url", "html"]
        sentence_tokens=[[word for word in sentence if word not in stop_words]for sentence in sentences]
        return sentence_tokens

    def textLemmatizer(self, sentences):
      lemmatizer = WordNetLemmatizer()
      lemmatized_sentences = [[lemmatizer.lemmatize(word) for word in sentence] for sentence in sentences]
      return lemmatized_sentences

### Summarization

In [ ]:
class frequencySummarizer:

    def summarize(self, article):
      freqTable = self.countWords(article)
      sentences, sentenceValue = self.wordInArticle(freqTable, article)

      sumValues = 0
      for sentence in sentenceValue:
        sumValues += sentenceValue[sentence]

      average = int(sumValues / len(sentenceValue))

      summary = ''

      for sentence in sentences:
        if (sentence in sentenceValue) and (sentenceValue[sentence] > (1.20 * average)):
          summary += " " + sentence

      return summary

    def countWords(self, article):
      stopWords = set(stopwords.words("english"))
      words = word_tokenize(article)
      freqTable = dict()

      for word in words:
        if word in stopWords:
          continue
        if word in freqTable:
          freqTable[word] += 1
        else:
          freqTable[word] = 1

      return freqTable

    def wordInArticle(self, freqTable, article):
      sentences = sent_tokenize(article)
      sentenceValue = dict()

      for sentence in sentences:
        for word, freq in freqTable.items():
          if word in sentence.lower():
              if sentence in sentenceValue:
                sentenceValue[sentence] += freq
              else:
                sentenceValue[sentence] = freq
      return sentences, sentenceValue

### Text Rank Summarization

In [ ]:
class textRankSummarizer:

    def textSummarize(self, sentences):
      cleaned_sentences = [" ".join(sentence) for sentence in sentences]
      sentence_embeddings = self.word2vec(sentences)
      similarity_matrix = self.similarity_matrix(sentence_embeddings)

      scores = self.pagerank(similarity_matrix)
      if len(scores) == 0:
        return " ".join(cleaned_sentences)

      doc_list = self.selectTopSentences(cleaned_sentences, scores)

      article = " ".join(doc_list)
      return article

    def word2vec(self, sentence):
        w2v=Word2Vec(sentence,vector_size=1,min_count=1,epochs=100, sg=0)
        sentence_embeddings=[[w2v.wv[word][0] for word in words] for words in sentence]
        max_len=max([len(tokens) for tokens in sentence])
        sentence_embeddings=[np.pad(embedding,(0,max_len-len(embedding)),'constant') for embedding in sentence_embeddings]
        return sentence_embeddings

    def similarity_matrix(self, sentence_embeddings):
        similarity_matrix = np.zeros([len(sentence_embeddings), len(sentence_embeddings)])
        for i,row_embedding in enumerate(sentence_embeddings):
            for j,column_embedding in enumerate(sentence_embeddings):
                similarity_matrix[i][j]=1-spatial.distance.cosine(row_embedding,column_embedding)
        return similarity_matrix

    def pagerank(self, similarity_matrix):
        nx_graph = nx.from_numpy_array(similarity_matrix)
        # Increase max_iter and add error handling
        try:
            scores = nx.pagerank(nx_graph)
        except nx.PowerIterationFailedConvergence:
            # Handle convergence failure, e.g., return empty scores
            # print("PageRank did not converge after 500 iterations.")
            scores = {}
        return scores

    def selectTopSentences(self, cleaned_sentences, scores):
        top_sentence={sentence:scores[index] for index, sentence in enumerate(cleaned_sentences)}
        n = len(top_sentence.items())
        top=dict(sorted(top_sentence.items(), key=lambda x: x[1], reverse=True)[:n])

        # find the median score and return any sentence that is scored greater median socre
        median_score = np.median(np.array(list(scores.values())))
        above_score_indices = np.array(list(top.values())) > median_score
        doc_list = list(np.array(list(top.keys()))[above_score_indices])
        return doc_list

## Topic Modeling

In [ ]:
# %%capture
# !pip install bertopic sentence_transformers

In [ ]:
# from sentence_transformers import SentenceTransformer
# from umap import UMAP
# from hdbscan import HDBSCAN
# from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, PartOfSpeech
# from bertopic import BERTopic

In [ ]:
# Pre-calculate embeddings
# embedding_model = SentenceTransformer("BAAI/bge-small-en")
# embeddings = embedding_model.encode(articles, show_progress_bar=True)

# umap_model = UMAP(n_neighbors = 15,
#                   n_components = 5,
#                   min_dist = 0.0,
#                   metric = "cosine",
#                   random_state=42)

# hdbscan_model = HDBSCAN(
#     min_cluster_size = 150,
#     metric = "euclidean",
#     cluster_selection_method = "eom",
#     prediction_data = True)


# top_n_keywords = 30

# pos_patterns = [
#             [{'POS': 'ADJ'}, {'POS': 'NOUN'}],
#             [{'POS': 'NOUN'}], [{'POS': 'ADJ'}]
# ]

# keybert = KeyBERTInspired(top_n_words=top_n_keywords)
# mmr = MaximalMarginalRelevance(top_n_words=top_n_keywords, diversity=0.3)
# pos = PartOfSpeech("en_core_web_sm", top_n_words=top_n_keywords, pos_patterns=pos_patterns)


# # All representation models
# representation_model = {
#     "KeyBERT": keybert,
#     "MMR": mmr,
#     "POS": pos
# }


# topic_model = BERTopic(
#     #sub-models
#     embedding_model = embedding_model,
#     umap_model = umap_model,
#     hdbscan_model = hdbscan_model,
#     representation_model = representation_model,

#     # Hyperparameters
#     n_gram_range=(1, 3),
#     verbose = True
# )

# topics, probs = topic_model.fit_transform(articles, embeddings)

In [ ]:
%%capture
!pip install groq

In [ ]:
from groq import Groq

In [ ]:
class GroqTopic:
    def __init__(self, model, api_key):
        self.client = Groq(api_key=api_key)
        self.model = model

        self.tech_classification_results = []
        # self.en_classification_results = []
        # self.translated_results = []
        self.labeling_results = []

        self.tech_classification_sys_prompt = """
                                        You are a helpful, respectful, and honest assistant for classifying topics.
                                        To classify whether a topic relates to technology, follow these guidelines:
                                        1. Analyze each keyword carefully to ensure the topic does not contains:
                                        - Consumer technology or consumer products (e.g., laptop, phone, earphone, camera, TV, earphone, smart home product, movie).
                                        - Error information on certain websites.
                                        - Any non-American English word.
                                        2. Consider the relationship between keywords to determine if the topic:
                                        - Describes electrical vehicles (acceptable).
                                        - Involves core technology concepts or advancements.
                                        3. Respond with only one word:
                                        - 'Yes' if the topic is related to Technology and follows the rules above. -
                                        'No' if the topic is not related to technology and does not follows the rules above. Do not provide any explanations or additional information in your response.
                                        """

        self.tech_summarization_sys_prompt = """
                                            You are a helpful, respectful and honest assistant for summarizing articles.
                                            You should summarize articles as more details as possible.
                                            Please only return summarized content without a title.
                                            """

        # self.en_classification_sys_prompt = """You are a helpful, respectful and honest assistant for classifying topics.
        #                                     Help me check if the entire article is written in American English words. Return only "Yes" or "No" nothing more."""

        # self.translation_sys_prompt = """You are a helpful, respectful and honest assistant for translating text.
        #                                 Help me translate an article into a English article. Please only return the translated article nothing more."""

        self.labeling_sys_prompt = """
                                You are a helpful, respectful and honest assistant for labeling topics.
                                Based on the keywords provided for a topic, generate a concise and specific label related to technology.
                                The label must be no longer than 5 words. Please return only the label, nothing else.
                                """

    # def fit_language(self, articles):
    #     for article in articles:
    #         result = self.is_english(article)
    #         result = result.lower()
    #         self.en_classification_results.append(result)

    #         if result.lower() == "no":
    #           self.translated_results.append(self.translate(article))
    #         else:
    #           self.translated_results.append(article)

    def fit_topic(self, keywords_list):
        for keywords in keywords_list:
          result = self.filter(keywords)
          self.tech_classification_results.append(result)

          if result.lower() == "yes":
            self.labeling_results.append(self.label(keywords))
          else:
            self.labeling_results.append("NA")

    # def get_translated_index(self):
    #   return np.where(np.array(self.en_classification_results) == "No")

    # def get_translated_results(self):
    #   return self.translated_results

    def get_topic_result(self):
      return pd.DataFrame({"KeyBert": self.keywords_list,
                           "TechClassification": self.classification_results,
                           "Label": self.labeling_results})

    # def get_Tech_classification(self):
    #   return self.tech_classification_sys_prompt

    # def get_topic_labeling(self):
    #   return self.labeling_results

    # def get_topic_index(self, index):
    #   return np.where(np.array(self.labeling_results) != "NA")[0]

    def modelInterface(self, sys_prompt, user_input, temperature):
      chat_completion = self.client.chat.completions.create(
          model=self.model,
          messages=[
              {
                  "role": "system",
                  "content": sys_prompt
                  },
              {
                  "role": "user",
                  "content": user_input,
              }
            ],
          temperature=1,
          max_tokens=8192,
          )
      return chat_completion.choices[0].message.content

    # def translate(self, text):
    #    return self.modelInterface(sys_prompt = self.translation_sys_prompt,
    #                               user_input = text,
    #                               temperature = 1)

    # def is_english(self, text):
    #   return self.modelInterface(sys_prompt = self.en_classification_sys_prompt,
    #                              user_input = text,
    #                              temperature = 1)

    def summarize(self, text):
      return self.modelInterface(sys_prompt = self.tech_summarization_sys_prompt,
                                 user_input = text,
                                 temperature = 1)

    def filter(self, keywords):
      return self.modelInterface(sys_prompt = self.tech_classification_sys_prompt,
                                 user_input = keywords,
                                 temperature = 1)

    def label(self, keywords):
      return self.modelInterface(sys_prompt = self.labeling_sys_prompt,
                                 user_input = keywords,
                                 temperature = 1.2)